# 📚 Company Knowledge Assistant — Enterprise RAG System

**Fully automated Google Colab notebook** that builds a production-quality,
GitHub-ready Retrieval Augmented Generation (RAG) project from scratch —
document loading, custom chunking, embeddings, FAISS vector search, a
Gemini-powered answer engine, conversation memory, a FastAPI backend, and a
Streamlit chat UI.

**How to use this notebook:**
1. Run every cell from top to bottom (`Runtime > Run all`).
2. Paste your **Gemini API key** (get one at https://aistudio.google.com/).
3. Upload documents via the FastAPI `/upload` endpoint, the Streamlit UI, or
   by dropping files into `data/uploaded_docs/` and re-running the testing cell.
4. At the end, the whole project is zipped and downloaded automatically —
   ready to push to GitHub.



## 🧩 Section 1 — Install Dependencies

In [1]:
%%capture
!pip install -q langchain==0.3.7 langchain-community==0.3.5 \
    google-generativeai==0.8.3 faiss-cpu==1.9.0 \
    sentence-transformers==3.3.1 fastapi==0.115.5 \
    "uvicorn[standard]==0.32.1" streamlit==1.40.1 pydantic==2.10.2 \
    pypdf==5.1.0 python-docx==1.1.2 python-multipart==0.0.17 \
    python-dotenv==1.0.1 pyngrok==7.2.1


In [2]:
print("✅ All dependencies installed.")


✅ All dependencies installed.


### 🔑 Configure Your Gemini API Key
This key is used everywhere in the project via the `GEMINI_API_KEY` environment variable, together with the model name `gemini-3.5-flash`.

In [3]:
import os
from getpass import getpass

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Paste your Gemini API key (input hidden): ").strip()

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
if not GEMINI_API_KEY:
    print("⚠️  No API key provided. You can set it later via os.environ['GEMINI_API_KEY'] = '...'")
else:
    print("✅ Gemini API key configured for this session.")

# Model name used everywhere in this project (change via GEMINI_MODEL env var if needed)
os.environ.setdefault("GEMINI_MODEL", "gemini-3.5-flash")
print(f"Using Gemini model: {os.environ['GEMINI_MODEL']}")


Paste your Gemini API key (input hidden): ··········
✅ Gemini API key configured for this session.
Using Gemini model: gemini-3.5-flash


## 📁 Section 2 — Create Project Folder Structure

In [4]:
import os

PROJECT_NAME = "company-knowledge-assistant"
BASE_DIR = os.path.join(os.getcwd(), PROJECT_NAME)

folders = [
    BASE_DIR,
    f"{BASE_DIR}/data",
    f"{BASE_DIR}/data/uploaded_docs",
    f"{BASE_DIR}/data/vector_store",
    f"{BASE_DIR}/src",
    f"{BASE_DIR}/api",
    f"{BASE_DIR}/streamlit_app",
    f"{BASE_DIR}/docs",
    f"{BASE_DIR}/docs/screenshots",
    f"{BASE_DIR}/notebooks",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Created: {folder}")

os.chdir(BASE_DIR)
print(f"\n📂 Working directory changed to: {os.getcwd()}")


Created: /content/company-knowledge-assistant
Created: /content/company-knowledge-assistant/data
Created: /content/company-knowledge-assistant/data/uploaded_docs
Created: /content/company-knowledge-assistant/data/vector_store
Created: /content/company-knowledge-assistant/src
Created: /content/company-knowledge-assistant/api
Created: /content/company-knowledge-assistant/streamlit_app
Created: /content/company-knowledge-assistant/docs
Created: /content/company-knowledge-assistant/docs/screenshots
Created: /content/company-knowledge-assistant/notebooks

📂 Working directory changed to: /content/company-knowledge-assistant


## 📝 Section 3 — Generate Project Files (README, requirements, config, boilerplate)

In [5]:
%%writefile README.md
# 📚 Company Knowledge Assistant

### Enterprise Retrieval Augmented Generation (RAG) System

[![Python](https://img.shields.io/badge/python-3.10%2B-blue.svg)](https://www.python.org/)
[![FastAPI](https://img.shields.io/badge/FastAPI-0.115-009688.svg)](https://fastapi.tiangolo.com/)
[![Streamlit](https://img.shields.io/badge/Streamlit-1.40-FF4B4B.svg)](https://streamlit.io/)
[![FAISS](https://img.shields.io/badge/FAISS-vector--search-4B8BBE.svg)](https://github.com/facebookresearch/faiss)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](LICENSE)
[![Made with Gemini](https://img.shields.io/badge/LLM-Google%20Gemini-4285F4.svg)](https://ai.google.dev/)

---

## 📖 Overview

**Company Knowledge Assistant** is a production-style, modular Retrieval Augmented
Generation (RAG) application that lets you upload internal company documents
(PDF, DOCX, TXT) and ask natural-language questions about them. Answers are
generated by **Google Gemini**, grounded strictly in the content you upload,
with source citations for every response.

The project ships with:

- A reusable Python RAG engine (`src/`)
- A **FastAPI** backend for programmatic access
- A **Streamlit** chat UI for end users
- Everything generated automatically from a single **Google Colab notebook**

---

## ✨ Features

- 📄 Multi-format document upload — PDF, DOCX, TXT
- 🧹 Document parsing & cleaning
- ✂️ Custom, configurable chunking (sentence-aware, not just fixed windows)
- 🧠 Embeddings via `sentence-transformers/all-MiniLM-L6-v2`
- 🗂️ Persistent FAISS vector store (save / load / update / search)
- 🔍 Top-K retrieval with similarity scoring and thresholding
- 💬 Conversation memory with a configurable history limit
- 📌 Source citations on every answer
- 🚫 Strict "answer only from context" guardrail — no hallucinated answers
- 🌐 FastAPI REST API (`/upload`, `/chat`, `/reset`, `/health`)
- 🎨 Streamlit chat interface with sidebar upload & source viewer
- 🐳 Docker-ready
- ☁️ 100% buildable from Google Colab — no local setup required

---

## 🏗️ Architecture

```
 Documents (PDF/DOCX/TXT)
        │
        ▼
   Document Loader
        │
        ▼
  Custom Text Splitter  (chunk_size / chunk_overlap)
        │
        ▼
  Sentence-Transformer Embeddings
        │
        ▼
     FAISS Vector Store
        │
        ▼
   Top-K Retriever (+ score filtering)
        │
        ▼
    Prompt Builder  (context-grounded)
        │
        ▼
     Google Gemini (gemini-3.5-flash)
        │
        ▼
  Answer + Sources + Memory
```

---

## 📂 Folder Structure

```
company-knowledge-assistant/
├── README.md
├── requirements.txt
├── .gitignore
├── Dockerfile
├── LICENSE
├── app.py                       # CLI entry point
├── data/
│   ├── uploaded_docs/            # Uploaded source documents
│   └── vector_store/             # Persisted FAISS index + metadata
├── src/
│   ├── loader.py                  # PDF / DOCX / TXT loading
│   ├── splitter.py                # Custom sentence-aware chunking
│   ├── embeddings.py              # Sentence-Transformers wrapper
│   ├── vectorstore.py             # FAISS persistence & search
│   ├── retriever.py               # Top-K retrieval + filtering
│   ├── prompt.py                  # Context-grounded prompt template
│   ├── rag_chain.py               # Full RAG pipeline orchestration
│   ├── chat_memory.py             # Bounded conversation memory
│   └── utils.py                   # Logging, config, Gemini client
├── api/
│   └── main.py                    # FastAPI application
├── streamlit_app/
│   └── app.py                     # Streamlit chat UI
├── docs/
│   └── screenshots/                # UI screenshots for this README
└── notebooks/
    └── CompanyKnowledgeAssistant.ipynb   # The notebook that built this project
```

---

## ⚙️ Installation

### Option A — Run in Google Colab (recommended)

1. Open `notebooks/CompanyKnowledgeAssistant.ipynb` in Google Colab.
2. Run every cell top to bottom.
3. Paste your Gemini API key when prompted (or set `GEMINI_API_KEY`).
4. Upload documents and start asking questions.

### Option B — Run locally

```bash
git clone https://github.com/<your-username>/company-knowledge-assistant.git
cd company-knowledge-assistant
python -m venv venv
source venv/bin/activate      # Windows: venv\Scripts\activate
pip install -r requirements.txt
export GEMINI_API_KEY="your-real-api-key"   # Windows: set GEMINI_API_KEY=...
```

---

## 🚀 Running Locally

### Start the FastAPI backend

```bash
uvicorn api.main:app --reload --port 8000
```

Visit interactive docs at `http://localhost:8000/docs`.

### Start the Streamlit UI

```bash
streamlit run streamlit_app/app.py
```

### Run the CLI chatbot

```bash
python app.py --docs data/uploaded_docs/handbook.pdf
```

### Run with Docker

```bash
docker build -t company-knowledge-assistant .
docker run -p 8000:8000 -e GEMINI_API_KEY="your-real-api-key" company-knowledge-assistant
```

---

## 🔌 API Reference

| Method | Endpoint   | Description                              |
|--------|------------|--------------------------------------------|
| GET    | `/health`  | Health check + vector store size           |
| POST   | `/upload`  | Upload and index PDF/DOCX/TXT documents    |
| POST   | `/chat`    | Ask a question, get a grounded answer      |
| POST   | `/reset`   | Clear conversation memory                  |

---

## 🖼️ Screenshots

> Add UI screenshots to `docs/screenshots/` and reference them here.

| Chat UI | Source Citations |
|---------|-------------------|
| ![chat](docs/screenshots/chat.png) | ![sources](docs/screenshots/sources.png) |

---

## 🗺️ Future Improvements

- [ ] Hybrid search (BM25 + dense retrieval)
- [ ] Streaming Gemini responses
- [ ] Multi-user authentication & per-user vector stores
- [ ] Support for additional formats (CSV, HTML, Markdown)
- [ ] Re-ranking with a cross-encoder
- [ ] Deployment templates for Render / Railway / GCP Cloud Run

---

## 🤝 Contributing

Contributions are welcome!

1. Fork the repository
2. Create a feature branch: `git checkout -b feature/my-feature`
3. Commit your changes: `git commit -m "Add my feature"`
4. Push and open a Pull Request

Please keep code PEP8-compliant, typed, and documented.

---

## 📄 License

This project is licensed under the [MIT License](LICENSE).

---

Built with ❤️ using LangChain, FAISS, Sentence-Transformers, FastAPI, Streamlit, and Google Gemini.


Overwriting README.md


In [6]:
%%writefile requirements.txt
langchain==0.3.7
langchain-community==0.3.5
google-generativeai==0.8.3
faiss-cpu==1.9.0
sentence-transformers==2.7.0
fastapi==0.115.5
uvicorn[standard]==0.32.1
streamlit==1.40.1
pydantic==2.10.2
pypdf==5.1.0
python-docx==1.1.2
python-multipart==0.0.17
python-dotenv==1.0.1
numpy==1.26.4
pyngrok==7.2.1

Overwriting requirements.txt


In [7]:
%%writefile .gitignore
__pycache__/
*.py[cod]
*.egg-info/
.env
.venv/
venv/
env/
data/uploaded_docs/*
!data/uploaded_docs/.gitkeep
data/vector_store/*
!data/vector_store/.gitkeep
.ipynb_checkpoints/
.DS_Store
*.log
*.zip
.pytest_cache/
.mypy_cache/


Overwriting .gitignore


In [8]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"]


Overwriting Dockerfile


In [9]:
%%writefile LICENSE
MIT License

Copyright (c) 2026 Company Knowledge Assistant Contributors

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.


Overwriting LICENSE


In [10]:
%%writefile src/__init__.py
"""Company Knowledge Assistant - core RAG package."""


Overwriting src/__init__.py


In [11]:
%%writefile api/__init__.py
"""FastAPI application package for the Company Knowledge Assistant."""


Overwriting api/__init__.py


In [12]:
%%writefile data/uploaded_docs/.gitkeep


Overwriting data/uploaded_docs/.gitkeep


In [13]:
%%writefile data/vector_store/.gitkeep


Overwriting data/vector_store/.gitkeep


In [14]:
%%writefile docs/screenshots/.gitkeep


Overwriting docs/screenshots/.gitkeep


## 📄 Section 4 — Document Loader (`src/loader.py`)
Supports PDF, DOCX, and TXT with dedicated error handling for missing, empty, or corrupted files.

In [15]:
%%writefile src/loader.py
"""
loader.py
=========
Document loading utilities for the Company Knowledge Assistant.

Supports loading and extracting raw text from PDF, DOCX, and TXT files.
"""

from __future__ import annotations

import logging
import os
from dataclasses import dataclass
from typing import List, Optional

from pypdf import PdfReader
from docx import Document as DocxDocument

logger = logging.getLogger(__name__)

SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".txt"}


class DocumentLoadError(Exception):
    """Raised when a document cannot be loaded or parsed."""


class UnsupportedFileTypeError(DocumentLoadError):
    """Raised when a file extension is not supported by the loader."""


@dataclass
class LoadedDocument:
    """Container for a loaded document's raw content and metadata."""

    filename: str
    filepath: str
    content: str
    file_type: str


class DocumentLoader:
    """
    Reusable loader for extracting text from PDF, DOCX, and TXT files.

    Example:
        loader = DocumentLoader()
        doc = loader.load("data/uploaded_docs/policy.pdf")
        docs = loader.load_multiple(["a.pdf", "b.docx", "c.txt"])
    """

    def __init__(self) -> None:
        logger.debug("DocumentLoader initialized.")

    def load(self, filepath: str) -> LoadedDocument:
        """Load a single document and return its extracted text + metadata."""
        if not os.path.exists(filepath):
            logger.error("File not found: %s", filepath)
            raise FileNotFoundError(f"File not found: {filepath}")

        if os.path.getsize(filepath) == 0:
            logger.error("File is empty: %s", filepath)
            raise DocumentLoadError(f"File is empty: {filepath}")

        ext = os.path.splitext(filepath)[1].lower()

        try:
            if ext == ".pdf":
                content = self._load_pdf(filepath)
            elif ext == ".docx":
                content = self._load_docx(filepath)
            elif ext == ".txt":
                content = self._load_txt(filepath)
            else:
                raise UnsupportedFileTypeError(
                    f"Unsupported file type '{ext}'. Supported types: {SUPPORTED_EXTENSIONS}"
                )
        except UnsupportedFileTypeError:
            raise
        except DocumentLoadError:
            raise
        except Exception as exc:  # noqa: BLE001
            logger.exception("Failed to load document: %s", filepath)
            raise DocumentLoadError(f"Failed to load '{filepath}': {exc}") from exc

        if not content or not content.strip():
            logger.warning("No extractable text found in: %s", filepath)

        return LoadedDocument(
            filename=os.path.basename(filepath),
            filepath=filepath,
            content=content.strip(),
            file_type=ext.replace(".", ""),
        )

    def load_multiple(self, filepaths: List[str]) -> List[LoadedDocument]:
        """Load several documents, skipping (and logging) any that fail."""
        documents: List[LoadedDocument] = []
        for filepath in filepaths:
            try:
                documents.append(self.load(filepath))
            except DocumentLoadError as exc:
                logger.error("Skipping file due to error: %s (%s)", filepath, exc)
                continue
        return documents

    @staticmethod
    def _load_pdf(filepath: str) -> str:
        try:
            reader = PdfReader(filepath)
        except Exception as exc:  # noqa: BLE001
            raise DocumentLoadError(f"Invalid or corrupted PDF file: {filepath}") from exc

        if reader.is_encrypted:
            try:
                reader.decrypt("")
            except Exception as exc:  # noqa: BLE001
                raise DocumentLoadError(f"Encrypted PDF could not be opened: {filepath}") from exc

        pages_text = []
        for page_number, page in enumerate(reader.pages, start=1):
            try:
                text = page.extract_text() or ""
            except Exception:  # noqa: BLE001
                logger.warning("Failed to extract text from page %s of %s", page_number, filepath)
                text = ""
            pages_text.append(text)

        return "\n".join(pages_text)

    @staticmethod
    def _load_docx(filepath: str) -> str:
        try:
            doc = DocxDocument(filepath)
        except Exception as exc:  # noqa: BLE001
            raise DocumentLoadError(f"Invalid or corrupted DOCX file: {filepath}") from exc

        paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]

        for table in doc.tables:
            for row in table.rows:
                row_text = " | ".join(cell.text.strip() for cell in row.cells)
                if row_text.strip(" |"):
                    paragraphs.append(row_text)

        return "\n".join(paragraphs)

    @staticmethod
    def _load_txt(filepath: str) -> str:
        encodings = ["utf-8", "latin-1"]
        last_error: Optional[Exception] = None
        for encoding in encodings:
            try:
                with open(filepath, "r", encoding=encoding) as file_handle:
                    return file_handle.read()
            except UnicodeDecodeError as exc:
                last_error = exc
                continue
        raise DocumentLoadError(f"Could not decode text file '{filepath}': {last_error}")


Overwriting src/loader.py


## ✂️ Section 5 — Custom Chunking (`src/splitter.py`)
A custom sentence-aware splitter with configurable `chunk_size` and `chunk_overlap` — not a bare LangChain default.

In [16]:
%%writefile src/splitter.py
"""
splitter.py
===========
Custom, configurable text chunking utilities.

Implements a sentence-aware character-based splitter so that chunk
boundaries prefer to fall on sentence edges rather than mid-word, while
still respecting a hard chunk_size limit and a configurable overlap.
This is a custom implementation and does not rely solely on LangChain's
default splitters.
"""

from __future__ import annotations

import logging
import re
from dataclasses import dataclass
from typing import List

logger = logging.getLogger(__name__)

_SENTENCE_BOUNDARY_RE = re.compile(r"(?<=[.!?])\s+")


@dataclass
class Chunk:
    """A single chunk of text with metadata about its source document."""

    text: str
    chunk_id: int
    source: str
    start_char: int
    end_char: int


class TextSplitter:
    """
    Custom sentence-aware text splitter.

    Sentences are packed greedily into chunks up to `chunk_size`
    characters. When a chunk is full, `chunk_overlap` trailing characters
    from the previous chunk are carried into the next one for context
    continuity.
    """

    def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 150) -> None:
        if chunk_size <= 0:
            raise ValueError("chunk_size must be a positive integer.")
        if chunk_overlap < 0:
            raise ValueError("chunk_overlap cannot be negative.")
        if chunk_overlap >= chunk_size:
            raise ValueError("chunk_overlap must be smaller than chunk_size.")

        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def _split_sentences(self, text: str) -> List[str]:
        text = re.sub(r"\s+", " ", text).strip()
        if not text:
            return []
        sentences = _SENTENCE_BOUNDARY_RE.split(text)
        return [s.strip() for s in sentences if s.strip()]

    def split_text(self, text: str) -> List[str]:
        """Split raw text into a list of chunk strings."""
        if not text or not text.strip():
            logger.warning("Attempted to split empty text.")
            return []

        sentences = self._split_sentences(text)
        chunks: List[str] = []
        current = ""

        for sentence in sentences:
            # Hard-split any single sentence longer than chunk_size.
            if len(sentence) > self.chunk_size:
                if current:
                    chunks.append(current.strip())
                    current = ""
                step = max(self.chunk_size - self.chunk_overlap, 1)
                for i in range(0, len(sentence), step):
                    piece = sentence[i:i + self.chunk_size]
                    chunks.append(piece.strip())
                continue

            candidate = f"{current} {sentence}".strip() if current else sentence

            if len(candidate) <= self.chunk_size:
                current = candidate
            else:
                if current:
                    chunks.append(current.strip())
                overlap_text = current[-self.chunk_overlap:] if self.chunk_overlap else ""
                current = f"{overlap_text} {sentence}".strip()

        if current:
            chunks.append(current.strip())

        return [c for c in chunks if c]

    def split_documents(self, documents: List) -> List[Chunk]:
        """
        Split a list of loaded-document objects (must expose `.content`
        and `.filename`) into a flat list of Chunk objects.
        """
        all_chunks: List[Chunk] = []
        for doc in documents:
            text_chunks = self.split_text(doc.content)
            cursor = 0
            for idx, chunk_text in enumerate(text_chunks):
                start = doc.content.find(chunk_text[:50], cursor) if chunk_text else -1
                start = max(start, 0)
                end = start + len(chunk_text)
                cursor = end
                all_chunks.append(
                    Chunk(
                        text=chunk_text,
                        chunk_id=idx,
                        source=doc.filename,
                        start_char=start,
                        end_char=end,
                    )
                )
            logger.info("Split '%s' into %d chunks.", doc.filename, len(text_chunks))
        return all_chunks


Overwriting src/splitter.py


## 🧠 Section 6 — Embeddings (`src/embeddings.py`)
Uses `sentence-transformers/all-MiniLM-L6-v2`.

In [17]:
%%writefile src/embeddings.py
"""
embeddings.py
=============
Embedding generation using Sentence-Transformers.
"""

from __future__ import annotations

import logging
from typing import List, Optional

import numpy as np
from sentence_transformers import SentenceTransformer

logger = logging.getLogger(__name__)

DEFAULT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


class EmbeddingError(Exception):
    """Raised when embedding generation fails."""


class EmbeddingModel:
    """
    Reusable wrapper around a Sentence-Transformers embedding model.

    The underlying model is loaded lazily on first use to keep notebook
    startup fast, and is cached on the instance afterwards.
    """

    def __init__(self, model_name: str = DEFAULT_MODEL_NAME) -> None:
        self.model_name = model_name
        self._model: Optional[SentenceTransformer] = None

    @property
    def model(self) -> SentenceTransformer:
        if self._model is None:
            logger.info("Loading embedding model: %s", self.model_name)
            try:
                self._model = SentenceTransformer(self.model_name)
            except Exception as exc:  # noqa: BLE001
                logger.exception("Failed to load embedding model.")
                raise EmbeddingError(
                    f"Could not load embedding model '{self.model_name}': {exc}"
                ) from exc
        return self._model

    def embed_documents(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        """Embed a list of texts and return a float32 numpy array."""
        if not texts:
            raise EmbeddingError("Cannot embed an empty list of texts.")
        try:
            embeddings = self.model.encode(
                texts,
                batch_size=batch_size,
                show_progress_bar=False,
                convert_to_numpy=True,
                normalize_embeddings=True,
            )
        except Exception as exc:  # noqa: BLE001
            logger.exception("Failed to embed documents.")
            raise EmbeddingError(f"Embedding generation failed: {exc}") from exc
        return embeddings.astype("float32")

    def embed_query(self, text: str) -> np.ndarray:
        """Embed a single query string."""
        if not text or not text.strip():
            raise EmbeddingError("Cannot embed an empty query string.")
        return self.embed_documents([text])[0]

    @property
    def dimension(self) -> int:
        return self.model.get_sentence_embedding_dimension()


Overwriting src/embeddings.py


## 🗂️ Section 7 — Vector Database (`src/vectorstore.py`)
FAISS-backed store with save / load / update / search.

In [18]:
%%writefile src/vectorstore.py
"""
vectorstore.py
==============
FAISS-backed vector store with save/load/update/search support.
"""

from __future__ import annotations

import json
import logging
import os
import pickle
from dataclasses import dataclass
from typing import List, Optional, Tuple

import faiss
import numpy as np

logger = logging.getLogger(__name__)

INDEX_FILENAME = "index.faiss"
METADATA_FILENAME = "metadata.pkl"


class VectorStoreError(Exception):
    """Raised for vector store initialization, persistence, or search errors."""


@dataclass
class ChunkMetadata:
    """Metadata stored alongside each vector in the FAISS index."""

    text: str
    source: str
    chunk_id: int


class FAISSVectorStore:
    """
    Thin, persistent wrapper around a FAISS flat inner-product index.

    Embeddings are expected to already be L2-normalized (see
    `EmbeddingModel`), so inner product search is equivalent to cosine
    similarity search.
    """

    def __init__(self, dimension: Optional[int] = None) -> None:
        self.dimension = dimension
        self.index: Optional[faiss.Index] = None
        self.metadata: List[ChunkMetadata] = []

        if dimension is not None:
            self._init_index(dimension)

    def _init_index(self, dimension: int) -> None:
        self.dimension = dimension
        self.index = faiss.IndexFlatIP(dimension)
        logger.info("Initialized FAISS IndexFlatIP with dimension=%d", dimension)

    def add(self, embeddings: np.ndarray, metadata_list: List[ChunkMetadata]) -> None:
        """Add new embeddings + metadata to the index (creates it if needed)."""
        if embeddings.shape[0] != len(metadata_list):
            raise VectorStoreError("Number of embeddings must match number of metadata entries.")

        if self.index is None:
            self._init_index(embeddings.shape[1])

        try:
            self.index.add(embeddings.astype("float32"))
        except Exception as exc:  # noqa: BLE001
            logger.exception("Failed to add embeddings to FAISS index.")
            raise VectorStoreError(f"Failed to add vectors to index: {exc}") from exc

        self.metadata.extend(metadata_list)
        logger.info("Added %d vectors. Total vectors: %d", len(metadata_list), self.index.ntotal)

    def search(self, query_embedding: np.ndarray, top_k: int = 5) -> List[Tuple[ChunkMetadata, float]]:
        """Return the top_k most similar (metadata, score) pairs."""
        if self.index is None or self.index.ntotal == 0:
            raise VectorStoreError("Vector store is empty or missing. Upload documents first.")

        query = np.asarray(query_embedding, dtype="float32").reshape(1, -1)
        top_k = min(top_k, self.index.ntotal)

        try:
            scores, indices = self.index.search(query, top_k)
        except Exception as exc:  # noqa: BLE001
            logger.exception("FAISS search failed.")
            raise VectorStoreError(f"Vector search failed: {exc}") from exc

        results: List[Tuple[ChunkMetadata, float]] = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            results.append((self.metadata[idx], float(score)))
        return results

    def save(self, directory: str) -> None:
        """Persist the FAISS index and metadata to disk."""
        if self.index is None:
            raise VectorStoreError("Cannot save an uninitialized vector store.")

        os.makedirs(directory, exist_ok=True)
        try:
            faiss.write_index(self.index, os.path.join(directory, INDEX_FILENAME))
            with open(os.path.join(directory, METADATA_FILENAME), "wb") as file_handle:
                pickle.dump(self.metadata, file_handle)
            with open(os.path.join(directory, "config.json"), "w", encoding="utf-8") as file_handle:
                json.dump({"dimension": self.dimension, "count": len(self.metadata)}, file_handle)
        except Exception as exc:  # noqa: BLE001
            logger.exception("Failed to save vector store.")
            raise VectorStoreError(f"Failed to save vector store to '{directory}': {exc}") from exc

        logger.info("Vector store saved to: %s", directory)

    def load(self, directory: str) -> None:
        """Load a previously persisted FAISS index and metadata from disk."""
        index_path = os.path.join(directory, INDEX_FILENAME)
        metadata_path = os.path.join(directory, METADATA_FILENAME)

        if not os.path.exists(index_path) or not os.path.exists(metadata_path):
            raise VectorStoreError(
                f"No vector store found at '{directory}'. Upload and process documents first."
            )

        try:
            self.index = faiss.read_index(index_path)
            with open(metadata_path, "rb") as file_handle:
                self.metadata = pickle.load(file_handle)
            self.dimension = self.index.d
        except Exception as exc:  # noqa: BLE001
            logger.exception("Failed to load vector store.")
            raise VectorStoreError(f"Failed to load vector store from '{directory}': {exc}") from exc

        logger.info("Vector store loaded from: %s (%d vectors)", directory, self.index.ntotal)

    def exists(self, directory: str) -> bool:
        """Check whether a persisted vector store exists at `directory`."""
        return os.path.exists(os.path.join(directory, INDEX_FILENAME))

    @property
    def count(self) -> int:
        return 0 if self.index is None else self.index.ntotal


Overwriting src/vectorstore.py


## 🔍 Section 8 — Retriever (`src/retriever.py`)
Top-K similarity search with optional score filtering.

In [19]:
%%writefile src/retriever.py
"""
retriever.py
============
Top-K similarity retrieval with optional score filtering.
"""

from __future__ import annotations

import logging
from typing import Dict, List, Optional

from .embeddings import EmbeddingModel
from .vectorstore import FAISSVectorStore, VectorStoreError

logger = logging.getLogger(__name__)


class Retriever:
    """
    Combines an embedding model and a FAISS vector store to retrieve the
    most relevant chunks for a given natural-language query.
    """

    def __init__(
        self,
        vector_store: FAISSVectorStore,
        embedding_model: EmbeddingModel,
        top_k: int = 5,
        score_threshold: Optional[float] = None,
    ) -> None:
        self.vector_store = vector_store
        self.embedding_model = embedding_model
        self.top_k = top_k
        self.score_threshold = score_threshold

    def retrieve(self, query: str, top_k: Optional[int] = None) -> List[Dict]:
        """Retrieve the most relevant chunks for `query` as a list of dicts."""
        if not query or not query.strip():
            logger.warning("Empty query passed to retriever.")
            return []

        k = top_k or self.top_k

        try:
            query_embedding = self.embedding_model.embed_query(query)
            results = self.vector_store.search(query_embedding, top_k=k)
        except VectorStoreError as exc:
            logger.warning("Retrieval failed: %s", exc)
            return []

        retrieved = []
        for metadata, score in results:
            if self.score_threshold is not None and score < self.score_threshold:
                continue
            retrieved.append(
                {
                    "text": metadata.text,
                    "source": metadata.source,
                    "chunk_id": metadata.chunk_id,
                    "score": round(score, 4),
                }
            )

        logger.info("Retrieved %d chunks for query: '%s'", len(retrieved), query[:60])
        return retrieved


Overwriting src/retriever.py


## 💬 Section 9 — Prompt Template (`src/prompt.py`)
Enforces context-only answers, with the required fallback message when information isn't found.

In [20]:
%%writefile src/prompt.py
"""
prompt.py
=========
Prompt construction for the RAG pipeline. Enforces context-grounded answers.
"""

from __future__ import annotations

from typing import Dict, List, Optional

FALLBACK_ANSWER = "I couldn't find that information in the uploaded documents."

SYSTEM_INSTRUCTIONS = f"""You are the Company Knowledge Assistant, an internal AI assistant that answers
questions strictly using the provided document context.

Rules you must always follow:
1. Answer ONLY using the information present in the "Context" section below.
2. Do not use outside knowledge, assumptions, or information not present in the context.
3. If the answer is not contained in the context, respond exactly with:
   "{FALLBACK_ANSWER}"
4. Be concise, accurate, and professional.
5. When useful, refer to the source document names provided in the context.
"""


def format_context(retrieved_chunks: List[Dict]) -> str:
    """Format retrieved chunks into a numbered context block with sources."""
    if not retrieved_chunks:
        return "No relevant context was found."

    lines = []
    for i, chunk in enumerate(retrieved_chunks, start=1):
        lines.append(
            f"[{i}] Source: {chunk['source']} (relevance: {chunk['score']})\n{chunk['text']}"
        )
    return "\n\n".join(lines)


def format_history(chat_history: List[Dict]) -> str:
    """Format prior turns of conversation for inclusion in the prompt."""
    if not chat_history:
        return "No previous conversation."

    lines = []
    for turn in chat_history:
        lines.append(f"User: {turn['question']}\nAssistant: {turn['answer']}")
    return "\n\n".join(lines)


def build_prompt(
    question: str,
    retrieved_chunks: List[Dict],
    chat_history: Optional[List[Dict]] = None,
) -> str:
    """Build the final prompt sent to Gemini."""
    context_block = format_context(retrieved_chunks)
    history_block = format_history(chat_history or [])

    return f"""{SYSTEM_INSTRUCTIONS}

Conversation history:
{history_block}

Context:
{context_block}

Question: {question}

Answer:"""


Overwriting src/prompt.py


## ✨ Section 10 — Gemini Integration (`src/utils.py`)
Centralized configuration and a resilient Gemini client (retries + clear error messages). Uses model `gemini-3.5-flash` everywhere.

In [21]:
%%writefile src/utils.py
"""
utils.py
========
Shared utilities: logging configuration, environment configuration, and the
Gemini API client wrapper.
"""

from __future__ import annotations

import logging
import os
import sys
import time
from dataclasses import dataclass
from typing import Optional

import google.generativeai as genai


# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------

def setup_logging(level: int = logging.INFO) -> None:
    """Configure root logging once for the whole application."""
    root_logger = logging.getLogger()
    if root_logger.handlers:
        return  # Already configured.

    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    handler.setFormatter(formatter)
    root_logger.addHandler(handler)
    root_logger.setLevel(level)


logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

@dataclass
class Settings:
    """Application configuration, sourced from environment variables."""

    gemini_api_key: str = os.getenv("GEMINI_API_KEY", "")
    gemini_model: str = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")
    chunk_size: int = int(os.getenv("CHUNK_SIZE", "1000"))
    chunk_overlap: int = int(os.getenv("CHUNK_OVERLAP", "150"))
    top_k: int = int(os.getenv("TOP_K", "5"))
    score_threshold: float = float(os.getenv("SCORE_THRESHOLD", "0.0"))
    memory_limit: int = int(os.getenv("MEMORY_LIMIT", "10"))
    vector_store_dir: str = os.getenv("VECTOR_STORE_DIR", "data/vector_store")
    upload_dir: str = os.getenv("UPLOAD_DIR", "data/uploaded_docs")


settings = Settings()


# ---------------------------------------------------------------------------
# Gemini client
# ---------------------------------------------------------------------------

class GeminiError(Exception):
    """Raised when the Gemini API cannot be reached or returns an error."""


class GeminiClient:
    """
    Thin wrapper around google-generativeai that adds retries, logging,
    and clear error messages for the RAG pipeline.
    """

    def __init__(
        self,
        api_key: Optional[str] = None,
        model_name: Optional[str] = None,
        max_retries: int = 2,
    ) -> None:
        self.api_key = api_key or settings.gemini_api_key
        self.model_name = model_name or settings.gemini_model
        self.max_retries = max_retries
        self._model = None

        if not self.api_key or self.api_key == "PASTE_YOUR_API_KEY_HERE":
            logger.warning("GEMINI_API_KEY is not set. Set it before making generation calls.")

    def _get_model(self):
        if self._model is None:
            try:
                genai.configure(api_key=self.api_key)
                self._model = genai.GenerativeModel(self.model_name)
            except Exception as exc:  # noqa: BLE001
                logger.exception("Failed to initialize Gemini model.")
                raise GeminiError(
                    f"Failed to initialize Gemini model '{self.model_name}': {exc}"
                ) from exc
        return self._model

    def generate(self, prompt: str, temperature: float = 0.2) -> str:
        """Generate a response from Gemini for the given prompt, with retries."""
        if not self.api_key or self.api_key == "PASTE_YOUR_API_KEY_HERE":
            raise GeminiError(
                "GEMINI_API_KEY is missing or is still the placeholder value. "
                "Set your real Gemini API key before chatting."
            )

        model = self._get_model()
        last_exception: Optional[Exception] = None

        for attempt in range(1, self.max_retries + 2):
            try:
                response = model.generate_content(
                    prompt,
                    generation_config={"temperature": temperature},
                )
                text = getattr(response, "text", None)
                if not text:
                    raise GeminiError("Gemini returned an empty response.")
                return text.strip()
            except Exception as exc:  # noqa: BLE001
                last_exception = exc
                logger.warning(
                    "Gemini call failed (attempt %d/%d): %s",
                    attempt, self.max_retries + 1, exc,
                )
                time.sleep(min(2 ** attempt, 8))

        logger.error("Gemini call failed after retries.")
        raise GeminiError(
            f"Gemini API call failed after {self.max_retries + 1} attempts: {last_exception}"
        )


Overwriting src/utils.py


## 🔗 Section 11 — RAG Chain (`src/rag_chain.py`)
Orchestrates: Documents → Chunking → Embeddings → Vector Store → Retriever → Prompt → Gemini → Answer.

In [22]:
%%writefile src/rag_chain.py
"""
rag_chain.py
============
End-to-end Retrieval Augmented Generation pipeline:

Documents -> Chunking -> Embeddings -> Vector Store -> Retriever -> Prompt -> Gemini -> Answer
"""

from __future__ import annotations

import logging
from typing import Dict, List, Optional

from .chat_memory import ConversationMemory
from .embeddings import EmbeddingModel
from .loader import DocumentLoader
from .prompt import FALLBACK_ANSWER, build_prompt
from .retriever import Retriever
from .splitter import TextSplitter
from .utils import GeminiClient, GeminiError, settings
from .vectorstore import ChunkMetadata, FAISSVectorStore, VectorStoreError

logger = logging.getLogger(__name__)


class RAGChain:
    """
    Orchestrates the full RAG pipeline: ingesting documents, building the
    vector store, and answering questions grounded in retrieved context.
    """

    def __init__(
        self,
        gemini_api_key: Optional[str] = None,
        chunk_size: int = settings.chunk_size,
        chunk_overlap: int = settings.chunk_overlap,
        top_k: int = settings.top_k,
        memory_limit: int = settings.memory_limit,
    ) -> None:
        self.loader = DocumentLoader()
        self.splitter = TextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        self.embedding_model = EmbeddingModel()
        self.vector_store = FAISSVectorStore()
        self.retriever = Retriever(self.vector_store, self.embedding_model, top_k=top_k)
        self.memory = ConversationMemory(max_length=memory_limit)
        self.gemini_client = GeminiClient(api_key=gemini_api_key)

    def ingest(self, filepaths: List[str]) -> int:
        """Load, chunk, embed, and index a list of document filepaths."""
        if not filepaths:
            raise ValueError("No files provided for ingestion.")

        documents = self.loader.load_multiple(filepaths)
        if not documents:
            raise ValueError("None of the provided files could be loaded.")

        chunks = self.splitter.split_documents(documents)
        if not chunks:
            raise ValueError("Document splitting produced no chunks (documents may be empty).")

        texts = [chunk.text for chunk in chunks]
        embeddings = self.embedding_model.embed_documents(texts)

        metadata_list = [
            ChunkMetadata(text=chunk.text, source=chunk.source, chunk_id=chunk.chunk_id)
            for chunk in chunks
        ]
        self.vector_store.add(embeddings, metadata_list)

        logger.info("Ingested %d documents into %d chunks.", len(documents), len(chunks))
        return len(chunks)

    def save_index(self, directory: str = settings.vector_store_dir) -> None:
        self.vector_store.save(directory)

    def load_index(self, directory: str = settings.vector_store_dir) -> None:
        self.vector_store.load(directory)

    def ask(self, question: str) -> Dict:
        """Answer a question using retrieved context and Gemini."""
        if not question or not question.strip():
            raise ValueError("Question cannot be empty.")

        try:
            retrieved_chunks = self.retriever.retrieve(question)
        except VectorStoreError as exc:
            logger.warning("Retrieval unavailable: %s", exc)
            retrieved_chunks = []

        if not retrieved_chunks:
            answer = FALLBACK_ANSWER
            sources: List[str] = []
        else:
            prompt = build_prompt(question, retrieved_chunks, self.memory.get_history())
            try:
                answer = self.gemini_client.generate(prompt)
            except GeminiError as exc:
                logger.error("Gemini generation failed: %s", exc)
                answer = f"An error occurred while generating the answer: {exc}"
            sources = sorted({chunk["source"] for chunk in retrieved_chunks})

        self.memory.add_exchange(question, answer)

        return {
            "answer": answer,
            "sources": sources,
            "retrieved_chunks": retrieved_chunks,
        }

    def reset(self) -> None:
        """Clear conversation memory (does not delete the vector store)."""
        self.memory.clear()


Overwriting src/rag_chain.py


## 🧵 Section 12 — Conversation Memory (`src/chat_memory.py`)
Stores previous Q&A pairs with a configurable, bounded history length.

In [23]:
%%writefile src/chat_memory.py
"""
chat_memory.py
==============
Simple bounded conversation memory for the RAG chatbot.
"""

from __future__ import annotations

import logging
from collections import deque
from typing import Deque, Dict, List

logger = logging.getLogger(__name__)


class ConversationMemory:
    """Stores the most recent question/answer pairs, up to a fixed limit."""

    def __init__(self, max_length: int = 10) -> None:
        if max_length <= 0:
            raise ValueError("max_length must be a positive integer.")
        self.max_length = max_length
        self._history: Deque[Dict[str, str]] = deque(maxlen=max_length)

    def add_exchange(self, question: str, answer: str) -> None:
        """Add a question/answer pair, evicting the oldest if over the limit."""
        self._history.append({"question": question, "answer": answer})
        logger.debug("Memory size: %d/%d", len(self._history), self.max_length)

    def get_history(self) -> List[Dict[str, str]]:
        return list(self._history)

    def clear(self) -> None:
        self._history.clear()
        logger.info("Conversation memory cleared.")

    def __len__(self) -> int:
        return len(self._history)


Overwriting src/chat_memory.py


## 🌐 Section 13 — FastAPI (`api/main.py`)
Endpoints: `/upload`, `/chat`, `/reset`, `/health`.

In [24]:
%%writefile api/main.py
"""
main.py
=======
FastAPI application exposing the Company Knowledge Assistant RAG pipeline.

Endpoints:
    POST /upload  - Upload one or more documents (PDF, DOCX, TXT) and index them.
    POST /chat    - Ask a question and receive a grounded answer with sources.
    POST /reset   - Clear conversation memory.
    GET  /health  - Health check.
"""

from __future__ import annotations

import logging
import os
import sys
from typing import List

from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from src.rag_chain import RAGChain  # noqa: E402
from src.utils import setup_logging, settings  # noqa: E402

setup_logging()
logger = logging.getLogger(__name__)

app = FastAPI(
    title="Company Knowledge Assistant API",
    description="Enterprise Retrieval Augmented Generation (RAG) API powered by Google Gemini.",
    version="1.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

rag_chain = RAGChain()


class ChatRequest(BaseModel):
    question: str = Field(..., min_length=1, description="The user's question.")


class ChatResponse(BaseModel):
    answer: str
    sources: List[str]


class UploadResponse(BaseModel):
    message: str
    files_processed: int
    chunks_indexed: int


class HealthResponse(BaseModel):
    status: str
    vector_store_size: int


@app.get("/health", response_model=HealthResponse)
def health() -> HealthResponse:
    """Simple health check endpoint."""
    return HealthResponse(status="ok", vector_store_size=rag_chain.vector_store.count)


@app.post("/upload", response_model=UploadResponse)
async def upload(files: List[UploadFile] = File(...)) -> UploadResponse:
    """Upload and index one or more PDF/DOCX/TXT documents."""
    if not files:
        raise HTTPException(status_code=400, detail="No files were uploaded.")

    os.makedirs(settings.upload_dir, exist_ok=True)
    saved_paths = []

    for uploaded_file in files:
        extension = os.path.splitext(uploaded_file.filename or "")[1].lower()
        if extension not in {".pdf", ".docx", ".txt"}:
            raise HTTPException(
                status_code=400,
                detail=f"Unsupported file type '{extension}'. Only PDF, DOCX, and TXT are allowed.",
            )

        destination = os.path.join(settings.upload_dir, uploaded_file.filename)
        try:
            contents = await uploaded_file.read()
            if not contents:
                raise HTTPException(status_code=400, detail=f"File '{uploaded_file.filename}' is empty.")
            with open(destination, "wb") as file_handle:
                file_handle.write(contents)
            saved_paths.append(destination)
        except HTTPException:
            raise
        except Exception as exc:  # noqa: BLE001
            logger.exception("Failed to save uploaded file: %s", uploaded_file.filename)
            raise HTTPException(status_code=500, detail=f"Failed to save file: {exc}") from exc

    try:
        chunks_indexed = rag_chain.ingest(saved_paths)
        rag_chain.save_index()
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    except Exception as exc:  # noqa: BLE001
        logger.exception("Ingestion failed.")
        raise HTTPException(status_code=500, detail=f"Failed to process documents: {exc}") from exc

    return UploadResponse(
        message="Files uploaded and indexed successfully.",
        files_processed=len(saved_paths),
        chunks_indexed=chunks_indexed,
    )


@app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest) -> ChatResponse:
    """Ask a question grounded in the uploaded documents."""
    try:
        result = rag_chain.ask(request.question)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    except Exception as exc:  # noqa: BLE001
        logger.exception("Chat request failed.")
        raise HTTPException(status_code=500, detail=f"Failed to answer question: {exc}") from exc

    return ChatResponse(answer=result["answer"], sources=result["sources"])


@app.post("/reset")
def reset() -> dict:
    """Clear conversation memory."""
    rag_chain.reset()
    return {"message": "Conversation memory has been reset."}


Overwriting api/main.py


### ▶️ Run the API (optional, inside Colab)

In [25]:
import subprocess
import time

# Launch FastAPI in the background (optional — useful for testing the API in Colab)
api_process = subprocess.Popen(
    ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(5)
print(f"🚀 FastAPI server starting on port 8000 (PID: {api_process.pid})")
print("\nTo expose it publicly from Colab, use pyngrok:\n")
print('    from pyngrok import ngrok')
print('    ngrok.set_auth_token("YOUR_NGROK_TOKEN")')
print('    public_url = ngrok.connect(8000)')
print('    print(public_url)')
print("\nOr just call it locally within Colab, e.g.:")
print('    import requests')
print('    requests.get("http://localhost:8000/health").json()')


🚀 FastAPI server starting on port 8000 (PID: 6962)

To expose it publicly from Colab, use pyngrok:

    from pyngrok import ngrok
    ngrok.set_auth_token("YOUR_NGROK_TOKEN")
    public_url = ngrok.connect(8000)
    print(public_url)

Or just call it locally within Colab, e.g.:
    import requests
    requests.get("http://localhost:8000/health").json()


## 🎨 Section 14 — Streamlit UI (`streamlit_app/app.py`) + CLI (`app.py`)
Sidebar upload, chat interface, conversation history, and retrieved-source display.

In [26]:
%%writefile streamlit_app/app.py
"""
streamlit_app/app.py
=====================
Streamlit UI for the Company Knowledge Assistant.
"""

from __future__ import annotations

import os
import sys

import streamlit as st

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from src.rag_chain import RAGChain  # noqa: E402
from src.utils import setup_logging, settings  # noqa: E402

setup_logging()

st.set_page_config(
    page_title="Company Knowledge Assistant",
    page_icon="📚",
    layout="wide",
)


@st.cache_resource(show_spinner=False)
def get_rag_chain(api_key: str) -> RAGChain:
    return RAGChain(gemini_api_key=api_key)


def init_session_state() -> None:
    if "messages" not in st.session_state:
        st.session_state.messages = []
    if "documents_indexed" not in st.session_state:
        st.session_state.documents_indexed = 0


def render_sidebar():
    st.sidebar.title("📚 Company Knowledge Assistant")
    st.sidebar.markdown("Enterprise RAG powered by **Google Gemini**.")

    api_key = st.sidebar.text_input(
        "Gemini API Key",
        value=os.getenv("GEMINI_API_KEY", ""),
        type="password",
        help="Paste your Gemini API key. Get one at https://aistudio.google.com/",
    )

    if not api_key:
        st.sidebar.warning("Enter your Gemini API key to get started.")
        return None

    rag_chain = get_rag_chain(api_key)

    st.sidebar.divider()
    st.sidebar.subheader("Upload documents")
    uploaded_files = st.sidebar.file_uploader(
        "PDF, DOCX, or TXT",
        type=["pdf", "docx", "txt"],
        accept_multiple_files=True,
    )

    if uploaded_files and st.sidebar.button("Process documents", use_container_width=True):
        os.makedirs(settings.upload_dir, exist_ok=True)
        saved_paths = []
        for uploaded_file in uploaded_files:
            destination = os.path.join(settings.upload_dir, uploaded_file.name)
            with open(destination, "wb") as file_handle:
                file_handle.write(uploaded_file.getbuffer())
            saved_paths.append(destination)

        with st.spinner("Indexing documents..."):
            try:
                chunks_indexed = rag_chain.ingest(saved_paths)
                rag_chain.save_index()
                st.session_state.documents_indexed += chunks_indexed
                st.sidebar.success(f"Indexed {chunks_indexed} chunks from {len(saved_paths)} file(s).")
            except Exception as exc:  # noqa: BLE001
                st.sidebar.error(f"Failed to process documents: {exc}")

    st.sidebar.divider()
    st.sidebar.metric("Chunks indexed", st.session_state.documents_indexed)

    if st.sidebar.button("Reset conversation", use_container_width=True):
        rag_chain.reset()
        st.session_state.messages = []
        st.sidebar.success("Conversation reset.")

    return rag_chain


def render_chat(rag_chain) -> None:
    st.title("💬 Ask your documents")

    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])
            if message.get("sources"):
                with st.expander("Sources"):
                    for source in message["sources"]:
                        st.markdown(f"- {source}")

    question = st.chat_input("Ask a question about your uploaded documents...")
    if question:
        st.session_state.messages.append({"role": "user", "content": question})
        with st.chat_message("user"):
            st.markdown(question)

        with st.chat_message("assistant"):
            with st.spinner("Thinking..."):
                try:
                    result = rag_chain.ask(question)
                    answer = result["answer"]
                    sources = result["sources"]
                except Exception as exc:  # noqa: BLE001
                    answer = f"An error occurred: {exc}"
                    sources = []

            st.markdown(answer)
            if sources:
                with st.expander("Sources"):
                    for source in sources:
                        st.markdown(f"- {source}")

        st.session_state.messages.append({"role": "assistant", "content": answer, "sources": sources})


def main() -> None:
    init_session_state()
    rag_chain = render_sidebar()

    if rag_chain is None:
        st.info("👋 Enter your Gemini API key in the sidebar to begin.")
        return

    render_chat(rag_chain)


if __name__ == "__main__":
    main()


Overwriting streamlit_app/app.py


In [27]:
%%writefile app.py
"""
app.py
======
Command-line entry point for the Company Knowledge Assistant.

Usage:
    python app.py --docs data/uploaded_docs/file1.pdf data/uploaded_docs/file2.docx

If no --docs are given, the app will try to load an existing vector store
from data/vector_store/, or ingest any files already sitting in
data/uploaded_docs/.
"""

from __future__ import annotations

import argparse
import glob
import os
import sys

from src.rag_chain import RAGChain
from src.utils import setup_logging, settings


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Company Knowledge Assistant CLI")
    parser.add_argument(
        "--docs",
        nargs="*",
        default=None,
        help="Paths to documents to ingest before chatting.",
    )
    return parser.parse_args()


def main() -> None:
    setup_logging()
    args = parse_args()

    rag_chain = RAGChain()

    docs = args.docs
    if not docs:
        docs = glob.glob(os.path.join(settings.upload_dir, "*"))

    if docs:
        print(f"Ingesting {len(docs)} document(s)...")
        try:
            chunks = rag_chain.ingest(docs)
            rag_chain.save_index()
            print(f"Indexed {chunks} chunks.")
        except Exception as exc:  # noqa: BLE001
            print(f"Failed to ingest documents: {exc}")
    elif rag_chain.vector_store.exists(settings.vector_store_dir):
        rag_chain.load_index()
        print("Loaded existing vector store.")
    else:
        print("No documents found. Add files to data/uploaded_docs/ or pass --docs.")
        sys.exit(1)

    print("\nCompany Knowledge Assistant is ready. Type 'exit' to quit.\n")
    while True:
        try:
            question = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if question.lower() in {"exit", "quit"}:
            print("Goodbye!")
            break
        if not question:
            continue

        result = rag_chain.ask(question)
        print(f"Assistant: {result['answer']}")
        if result["sources"]:
            print(f"Sources: {', '.join(result['sources'])}")
        print()


if __name__ == "__main__":
    main()


Overwriting app.py


### ▶️ Run the Streamlit app (optional, inside Colab)

In [28]:
print("To run the Streamlit UI from this Colab notebook:\n")
print("1) Start it in the background:")
print('   !streamlit run streamlit_app/app.py --server.port 8501 &>/content/logs.txt &\n')
print("2) Expose port 8501 publicly with pyngrok:")
print('   from pyngrok import ngrok')
print('   ngrok.set_auth_token("YOUR_NGROK_TOKEN")')
print('   public_url = ngrok.connect(8501)')
print('   print(public_url)\n')
print("Then open the printed public_url to use the chat UI.")


To run the Streamlit UI from this Colab notebook:

1) Start it in the background:
   !streamlit run streamlit_app/app.py --server.port 8501 &>/content/logs.txt &

2) Expose port 8501 publicly with pyngrok:
   from pyngrok import ngrok
   ngrok.set_auth_token("YOUR_NGROK_TOKEN")
   public_url = ngrok.connect(8501)
   print(public_url)

Then open the printed public_url to use the chat UI.


In [34]:
!streamlit run streamlit_app/app.py --server.port 8501 > streamlit.log 2>&1 &

In [35]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

--2026-08-13 09:53:12--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.7.3/cloudflared-linux-amd64 [following]
--2026-08-13 09:53:13--  https://github.com/cloudflare/cloudflared/releases/download/2026.7.3/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/3812f7fa-ce13-4147-a9fb-197be83d49fa?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-08-13T10%3A36%3A47Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-08-

In [36]:
!./cloudflared tunnel --url http://localhost:8501

2026-08-13T09:53:24Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-13T09:53:24Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-13T09:53:27Z INF +--------------------------------------------------------------------------------------------+
2026-08-13T09:53:27Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-13T09:53:27Z INF |  https://beginners-tag-weeks-dealers.trycloudflare.com

## ✅ Section 15 — Testing
End-to-end smoke tests across every module: loader → splitter → embeddings → vector store → retriever → memory → full RAG chain.

In [29]:
import sys
import os
import logging

sys.path.append(os.getcwd())

from src.loader import DocumentLoader
from src.splitter import TextSplitter
from src.embeddings import EmbeddingModel
from src.vectorstore import FAISSVectorStore, ChunkMetadata
from src.retriever import Retriever
from src.chat_memory import ConversationMemory
from src.rag_chain import RAGChain
from src.utils import setup_logging

setup_logging(logging.INFO)

print("=" * 60)
print("RUNNING SMOKE TESTS")
print("=" * 60)

sample_text = (
    "Company Knowledge Assistant is an internal RAG tool. "
    "Employees can upload PDF, DOCX, and TXT files. "
    "The assistant answers questions using only uploaded content. "
    "If information is missing, it says so honestly."
)

os.makedirs("data/uploaded_docs", exist_ok=True)
sample_path = "data/uploaded_docs/sample_policy.txt"
with open(sample_path, "w", encoding="utf-8") as f:
    f.write(sample_text)

# 1. Loader
loader = DocumentLoader()
doc = loader.load(sample_path)
assert doc.content.strip(), "Loader test failed: no content extracted."
print(f"[PASS] Loader: extracted {len(doc.content)} characters from {doc.filename}")

# 2. Splitter
splitter = TextSplitter(chunk_size=80, chunk_overlap=20)
chunks = splitter.split_documents([doc])
assert len(chunks) > 0, "Splitter test failed: no chunks produced."
print(f"[PASS] Splitter: produced {len(chunks)} chunks")

# 3. Embeddings
embedder = EmbeddingModel()
embeddings = embedder.embed_documents([c.text for c in chunks])
assert embeddings.shape[0] == len(chunks), "Embedding test failed: shape mismatch."
print(f"[PASS] Embeddings: shape = {embeddings.shape}")

# 4. Vector store
store = FAISSVectorStore()
metadata_list = [ChunkMetadata(text=c.text, source=c.source, chunk_id=c.chunk_id) for c in chunks]
store.add(embeddings, metadata_list)
results = store.search(embedder.embed_query("What file types are supported?"), top_k=2)
assert len(results) > 0, "Vector store test failed: no search results."
print(f"[PASS] Vector store: {store.count} vectors indexed, search returned {len(results)} result(s)")

# 5. Retriever
retriever = Retriever(store, embedder, top_k=2)
retrieved = retriever.retrieve("What file types are supported?")
assert len(retrieved) > 0, "Retriever test failed."
print(f"[PASS] Retriever: returned {len(retrieved)} relevant chunk(s)")

# 6. Memory
memory = ConversationMemory(max_length=2)
memory.add_exchange("q1", "a1")
memory.add_exchange("q2", "a2")
memory.add_exchange("q3", "a3")
assert len(memory) == 2, "Memory test failed: limit not enforced."
print("[PASS] Conversation memory: bounded correctly")

# 7. Full RAG chain (Gemini call needs a real API key to succeed)
rag_chain = RAGChain()
rag_chain.ingest([sample_path])
result = rag_chain.ask("What file types does the assistant support?")
print(f"[INFO] RAG chain answer: {result['answer'][:200]}")
print(f"[INFO] Sources: {result['sources']}")

print("=" * 60)
print("ALL SMOKE TESTS COMPLETED")
print("=" * 60)
print("\nNote: If GEMINI_API_KEY is missing or invalid, the final answer above")
print("will show a Gemini error message — this is expected until you provide")
print("a real, working Gemini API key.")


RUNNING SMOKE TESTS
[PASS] Loader: extracted 208 characters from sample_policy.txt
[PASS] Splitter: produced 4 chunks


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[PASS] Embeddings: shape = (4, 384)
[PASS] Vector store: 4 vectors indexed, search returned 2 result(s)
[PASS] Retriever: returned 2 relevant chunk(s)
[PASS] Conversation memory: bounded correctly
[INFO] RAG chain answer: According to `sample_policy.txt`, the assistant supports PDF, DOCX, and TXT files.
[INFO] Sources: ['sample_policy.txt']
ALL SMOKE TESTS COMPLETED

Note: If GEMINI_API_KEY is missing or invalid, the final answer above
will show a Gemini error message — this is expected until you provide
a real, working Gemini API key.


## 💾 Section 16 — Save Project
Persist a copy of this notebook into `notebooks/CompanyKnowledgeAssistant.ipynb` so it ships as part of the repo.

In [30]:
import json
import os

notebook_path = "notebooks/CompanyKnowledgeAssistant.ipynb"

try:
    from google.colab import _message
    notebook_json = _message.blocking_request("get_ipynb", request="", timeout_sec=30)
    with open(notebook_path, "w", encoding="utf-8") as f:
        json.dump(notebook_json["ipynb"], f, indent=1)
    print(f"✅ Notebook saved to {notebook_path}")
except Exception as exc:
    print("⚠️  Could not auto-save the notebook (not running in Colab, or API unavailable).")
    print("    Use File > Save a copy in Drive/GitHub, or manually place a copy of this")
    print(f"    notebook at: {notebook_path}")
    print(f"    Details: {exc}")


✅ Notebook saved to notebooks/CompanyKnowledgeAssistant.ipynb


## 📦 Section 17 — Zip Project
Zip the entire `company-knowledge-assistant/` folder and download it — ready to push to GitHub.

In [31]:
import shutil
import os

project_dir = os.getcwd()
zip_base_name = "company-knowledge-assistant"
zip_output_dir = os.path.dirname(project_dir)

zip_path = shutil.make_archive(
    base_name=os.path.join(zip_output_dir, zip_base_name),
    format="zip",
    root_dir=zip_output_dir,
    base_dir=os.path.basename(project_dir),
)

print(f"✅ Project zipped to: {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("⚠️  Not running in Colab, or the download API is unavailable.")
    print(f"    You can find the zip file at: {zip_path}")


✅ Project zipped to: /content/company-knowledge-assistant.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🎉 Done — Final Project Structure

In [32]:
print("📁 Final project structure:\n")
for root, dirs, files_ in os.walk("."):
    dirs[:] = [d for d in dirs if d not in {".git", "__pycache__"}]
    level = root.replace(".", "", 1).count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root) or '.'}/")
    for f in sorted(files_):
        print(f"{indent}    {f}")


📁 Final project structure:

./
    .gitignore
    Dockerfile
    LICENSE
    README.md
    app.py
    requirements.txt
    src/
        __init__.py
        chat_memory.py
        embeddings.py
        loader.py
        prompt.py
        rag_chain.py
        retriever.py
        splitter.py
        utils.py
        vectorstore.py
    api/
        __init__.py
        main.py
    docs/
        screenshots/
            .gitkeep
    company-knowledge-assistant/
        .gitignore
        Dockerfile
        LICENSE
        README.md
        app.py
        requirements.txt
        src/
            __init__.py
            chat_memory.py
            embeddings.py
            loader.py
            prompt.py
            rag_chain.py
            retriever.py
            splitter.py
            utils.py
            vectorstore.py
        api/
            __init__.py
            main.py
        docs/
            screenshots/
                .gitkeep
        streamlit_app/
            app.py
        